In [0]:
%pip install nflreadpy

In [0]:
dbutils.library.restartPython()

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Explore (weekly): 2026 offense, defense, and efficiency, v5
# MAGIC Runs every week after **ingest_raw** in the scheduled job. Reads only from `workspace.raw.*`.
# MAGIC
# MAGIC **Three rankings on every main table** (all backtested on 2018–2025):
# MAGIC - `raw_rank` = **Results**: plain EPA on every play (what happened; comparable to SumerSports / rbsdm)
# MAGIC - `rank` = **Quality**: garbage time down-weighted (not cut) + opponent-adjusted
# MAGIC - `forecast_rank` = **Forecast**: EPA + success rate, blended with last season as a prior
# MAGIC
# MAGIC **New in v5:** soft garbage-time weighting instead of a hard 10–90% cutoff, a prior-season
# MAGIC forecast, and rank ranges (`rank_low`–`rank_high`) from resampling whole games.
# MAGIC
# MAGIC Saves to `workspace.analytics`: `def_rankings_2026`, `def_pass_2026`, `def_rush_2026`,
# MAGIC `off_rankings_2026`, `team_efficiency_2026`, `team_games_2026`

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Load the season and build the shared play tables

# COMMAND ----------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEASON = 2026

# Garbage time is DOWN-WEIGHTED, not removed. Plays with win probability 20-80% count fully;
# weight tapers to GT_FLOOR as the game gets lopsided. A hard 10-90% cutoff predicted worst in
# the 2018-2025 backtest, and it threw away real dominance (e.g. leading big and still scoring).
GT_FLOOR = 0.3


def gt_weight(wp):
    dist = np.minimum(wp, 1 - wp)
    return np.clip(GT_FLOOR + (1 - GT_FLOOR) * (dist - 0.02) / 0.18, GT_FLOOR, 1.0)


def wmean(df, key, col="epa"):
    """Garbage-time-weighted average of `col` by `key` (team)."""
    x = df[col].astype(float) * df["w"]
    return x.groupby(df[key]).sum() / df["w"].groupby(df[key]).sum()

cols = ["season", "week", "game_id", "play_id", "posteam", "defteam", "play_type", "season_type",
        "down", "wp", "epa", "success", "pass", "rush", "qb_dropback", "yards_gained",
        "sack", "qb_hit", "interception", "fumble_lost",
        "series", "series_success", "series_result",
        "penalty", "penalty_team", "penalty_yards", "two_point_attempt"]

# This season plus last season (last season feeds the forecast prior)
pbp_all = (spark.table("workspace.raw.pbp")
           .filter(f"season in ({SEASON}, {SEASON - 1})")
           .select(*cols)
           .toPandas())
pbp_all = pbp_all[pbp_all["season_type"] == "REG"]
pbp = pbp_all[pbp_all["season"] == SEASON].copy()
pbp_prev = pbp_all[pbp_all["season"] == SEASON - 1].copy()

# All real offensive plays (used where garbage time doesn't matter, e.g. pressure rate).
# Standard nflfastR/rbsdm filter: pass or rush flag, so penalty plays like defensive pass
# interference and holding COUNT (play_type 'no_play' alone would drop them). 2-pt tries excluded.
all_plays = pbp[((pbp["pass"] == 1) | (pbp["rush"] == 1))
                & pbp["epa"].notna() & (pbp["two_point_attempt"] != 1)].copy()
all_plays["explosive"] = (
    ((all_plays["rush"] == 1) & (all_plays["yards_gained"] >= 10))
    | ((all_plays["pass"] == 1) & (all_plays["yards_gained"] >= 20))
)

# Every play counts, weighted by game state (garbage time down-weighted to GT_FLOOR)
all_plays["w"] = gt_weight(all_plays["wp"].fillna(0.5).to_numpy())
plays = all_plays
early = plays[plays["down"].isin([1, 2])]

# Phases: pass = dropbacks incl. scrambles, sacks, and pass penalties (DPI etc.);
# rush = designed runs only
pass_plays = plays[plays["pass"] == 1]
rush_plays = plays[plays["rush"] == 1]

WEEK = int(pbp["week"].max())
GAMES_OFF = pbp.groupby("posteam")["game_id"].nunique()
GAMES_DEF = pbp.groupby("defteam")["game_id"].nunique()
print(f"{SEASON} through week {WEEK}: {len(all_plays):,} plays "
      f"({(all_plays['w'] < 1).mean():.0%} down-weighted as garbage time)")


def save_table(pdf, name):
    """Save a pandas table to workspace.analytics for the dashboard."""
    (spark.createDataFrame(pdf)
         .write.mode("overwrite")
         .option("overwriteSchema", "true")
         .saveAsTable(f"workspace.analytics.{name}"))
    print(f"saved workspace.analytics.{name}")


spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.analytics")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Shared building blocks: opponent adjustment, series conversion, penalties

# COMMAND ----------

def opponent_adjust(df, K):
    """
    Opponent-adjusted EPA/play for every offense and defense in `df`.

    For each game, compare what happened to what the opponent USUALLY does in its
    other games (the game itself is excluded so a team doesn't adjust against itself).
    K "prior" plays pull each opponent's usual level toward league average, so a few
    fluky early-season games can't swing the rankings. The effect fades as data grows.

    Returns (adj_off, adj_def, per_game_table). Lower adj_def = better defense.
    """
    league = (df["epa"] * df["w"]).sum() / df["w"].sum()
    games = (df.assign(ew=df["epa"] * df["w"])
             .groupby(["game_id", "week", "posteam", "defteam"])
             .agg(epa_sum=("ew", "sum"), n=("w", "sum")).reset_index())
    off_tot = games.groupby("posteam")[["epa_sum", "n"]].sum()
    def_tot = games.groupby("defteam")[["epa_sum", "n"]].sum()

    g = (games.join(off_tot, on="posteam", rsuffix="_off")
              .join(def_tot, on="defteam", rsuffix="_def"))
    g["off_usual"] = (g["epa_sum_off"] - g["epa_sum"] + K * league) / (g["n_off"] - g["n"] + K)
    g["def_usual"] = (g["epa_sum_def"] - g["epa_sum"] + K * league) / (g["n_def"] - g["n"] + K)
    g["game_epa"] = g["epa_sum"] / g["n"]
    g["def_adj"] = g["game_epa"] - g["off_usual"]   # held this offense below its norm?
    g["off_adj"] = g["game_epa"] - g["def_usual"]   # beat this defense's norm?

    def weighted(col, key):
        return (g.groupby(key).apply(lambda d: (d[col] * d["n"]).sum() / d["n"].sum()) + league)

    return weighted("off_adj", "posteam"), weighted("def_adj", "defteam"), g


# Overall uses K=150; each phase has roughly half the plays, so it uses K=75
adj_off_epa, adj_def_epa, g = opponent_adjust(plays, K=150)
adj_off_pass, adj_def_pass, _ = opponent_adjust(pass_plays, K=75)
adj_off_rush, adj_def_rush, _ = opponent_adjust(rush_plays, K=75)
league_epa = (plays["epa"] * plays["w"]).sum() / plays["w"].sum()


# --- Forecast: this season's EPA + success rate, blended with last season as a prior ---
# Fit on 2018-2025 (leave-one-season-out): improved rest-of-season prediction in 6-7 of 8 seasons.
# Prior weight = K / (plays + K). Defense regresses more, so it leans on the prior longer.
FORECAST = {"def": dict(K=800, rho=0.3, beta=0.5),   # rho: share of last season's edge carried over
            "off": dict(K=500, rho=0.3, beta=0.5)}   # beta: weight on success rate vs EPA


def composite(df, key, beta):
    """Weighted EPA + beta * weighted success rate, centered on that season's league average."""
    epa, succ, n = wmean(df, key, "epa"), wmean(df, key, "success"), df["w"].groupby(df[key]).sum()
    return (epa - epa.mean()) + beta * (succ - succ.mean()), n, epa


prev = pbp_prev[((pbp_prev["pass"] == 1) | (pbp_prev["rush"] == 1))
                & pbp_prev["epa"].notna() & (pbp_prev["two_point_attempt"] != 1)].copy()
prev["w"] = gt_weight(prev["wp"].fillna(0.5).to_numpy())


def forecast(side):
    key = "defteam" if side == "def" else "posteam"
    p = FORECAST[side]
    cur, n, _ = composite(plays, key, p["beta"])
    pri, _, prior_epa = composite(prev, key, p["beta"])
    pri = pri.reindex(cur.index).fillna(0)
    score = (n * cur + p["K"] * p["rho"] * pri) / (n + p["K"])
    return score, prior_epa.reindex(cur.index), p["K"] / (n + p["K"])


def_forecast, def_prior_epa, def_prior_wt = forecast("def")
off_forecast, off_prior_epa, off_prior_wt = forecast("off")


# --- Rank ranges: resample whole games 1,000 times to show how uncertain each rank is ---
def rank_range(df, key, value, ascending, sims=1000, seed=0):
    """10th-90th percentile rank and top-10 probability, shifting `value` by game-resampling noise."""
    rng = np.random.default_rng(seed)
    gm = (df.assign(ew=df["epa"] * df["w"]).groupby([key, "game_id"])
          .agg(s=("ew", "sum"), n=("w", "sum")).reset_index())
    base = gm.groupby(key)["s"].sum() / gm.groupby(key)["n"].sum()
    sims_tbl = {}
    for team, t in gm.groupby(key):
        s, n = t["s"].to_numpy(), t["n"].to_numpy()
        idx = rng.integers(0, len(s), size=(sims, len(s)))
        sims_tbl[team] = s[idx].sum(1) / n[idx].sum(1) - base[team]
    noise = pd.DataFrame(sims_tbl)
    ranks = (noise + value.reindex(noise.columns)).rank(axis=1, ascending=ascending)
    return pd.DataFrame({"rank_low": ranks.quantile(0.1).round().astype(int),
                         "rank_high": ranks.quantile(0.9).round().astype(int),
                         "top10_pct": (ranks <= 10).mean().round(2)})

# --- Series conversion rate: share of series that earn a first down or TD ---
series = (pbp[pbp["posteam"].notna() & pbp["series"].notna()
              & ~pbp["series_result"].isin(["QB kneel", "End of half"])]
          .drop_duplicates(["game_id", "posteam", "series"])
          [["game_id", "posteam", "defteam", "series_success"]])
off_series = series.groupby("posteam")["series_success"].mean()
def_series = series.groupby("defteam")["series_success"].mean()

# --- Penalties per game (accepted fouls, every play type) ---
pens = pbp[(pbp["penalty"] == 1) & pbp["penalty_team"].notna()]
off_pens = pens[pens["penalty_team"] == pens["posteam"]]
def_pens = pens[pens["penalty_team"] == pens["defteam"]]
off_pen_pg = off_pens.groupby("posteam").size() / GAMES_OFF
off_pen_yds_pg = off_pens.groupby("posteam")["penalty_yards"].sum() / GAMES_OFF
def_pen_pg = def_pens.groupby("defteam").size() / GAMES_DEF
def_pen_yds_pg = def_pens.groupby("defteam")["penalty_yards"].sum() / GAMES_DEF

# --- Charting data used by both pass and rush defense ---
pfr = (spark.table("workspace.raw.pfr_def")
       .filter(f"season = {SEASON} and game_type = 'REG'")
       .select("team", "def_pressures", "def_targets", "def_completions_allowed",
               "def_yards_allowed", "def_missed_tackles", "def_tackles_combined")
       .toPandas()
       .groupby("team").sum())

ftn = (spark.table("workspace.raw.ftn_charting")
       .filter(f"season = {SEASON}")
       .select("nflverse_game_id", "nflverse_play_id", "n_blitzers", "n_pass_rushers", "n_defense_box")
       .toPandas())

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Overall defensive rankings
# MAGIC `rank` = quality (opponent-adjusted, garbage time down-weighted). `raw_rank` = results
# MAGIC (plain EPA, every play). `forecast_rank` = prediction (blended with last season). Lower = better.

# COMMAND ----------

defense = plays.groupby("defteam").agg(
    plays=("epa", "size"),
    epa_per_play=("epa", "mean"),          # plain EPA, every play (Results view)
)
defense["weighted_epa"] = wmean(plays, "defteam")
defense["success_rate"] = wmean(plays, "defteam", "success")
defense["explosive_rate"] = wmean(plays, "defteam", "explosive")
defense["adj_epa_per_play"] = adj_def_epa
defense["forecast_score"] = def_forecast   # centered on league average; lower = better
defense["prior_epa"] = def_prior_epa       # last season's EPA allowed
defense["prior_weight"] = def_prior_wt     # how much the forecast is pulled toward last season / league average
defense["adj_pass_epa"] = adj_def_pass
defense["adj_rush_epa"] = adj_def_rush
defense["pass_rank"] = defense["adj_pass_epa"].rank(method="min").astype(int)
defense["rush_rank"] = defense["adj_rush_epa"].rank(method="min").astype(int)
defense["early_down_epa"] = wmean(early, "defteam")
defense["early_down_success"] = wmean(early, "defteam", "success")
defense["series_conv_allowed"] = def_series

# Pass rush summary (all game situations). Full pass-rush and coverage detail is in def_pass_2026.
db_all = all_plays[all_plays["pass"] == 1]
defense["pressure_rate"] = pfr["def_pressures"] / db_all.groupby("defteam").size()
defense["sack_rate"] = db_all.groupby("defteam")["sack"].mean()
_ch = db_all.merge(ftn, left_on=["game_id", "play_id"], right_on=["nflverse_game_id", "nflverse_play_id"])
defense["blitz_rate"] = (_ch["n_blitzers"] > 0).groupby(_ch["defteam"]).mean()
defense["avg_rushers"] = _ch.groupby("defteam")["n_pass_rushers"].mean()
defense["no_blitz_epa"] = _ch[_ch["n_blitzers"] == 0].groupby("defteam")["epa"].mean()
defense["qb_hit_rate"] = db_all.groupby("defteam")["qb_hit"].mean()

# Raw (unadjusted) pass and rush EPA allowed, kept from v2 alongside the adjusted versions
defense["pass_epa"] = wmean(pass_plays, "defteam")
defense["rush_epa"] = wmean(rush_plays, "defteam")

# Context only (noisy week to week, so not used for ranking)
defense["sacks"] = all_plays.groupby("defteam")["sack"].sum()
defense["takeaways"] = (all_plays.groupby("defteam")["interception"].sum()
                        + all_plays.groupby("defteam")["fumble_lost"].sum())
defense["penalties_pg"] = def_pen_pg
defense["penalty_yds_pg"] = def_pen_yds_pg

defense = defense.sort_values("adj_epa_per_play")
defense.insert(0, "rank", range(1, len(defense) + 1))
defense.insert(1, "raw_rank", defense["epa_per_play"].rank(method="min").astype(int))
defense.insert(2, "forecast_rank", defense["forecast_score"].rank(method="min").astype(int))
defense = defense.join(rank_range(plays, "defteam", defense["adj_epa_per_play"], ascending=True))
defense.index.name = "team"

def_out = defense.round(3).reset_index()
def_out["through_week"] = WEEK

print(f"{SEASON} overall defensive rankings through week {WEEK}")
display(def_out)
save_table(def_out, f"def_rankings_{SEASON}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Pass defense
# MAGIC Ranked by **opponent-adjusted EPA per dropback** (lower = better).
# MAGIC Split into **pass rush** (pressure, sacks, blitz) and **coverage** (throws only, PFR target stats).

# COMMAND ----------

pass_d = pass_plays.groupby("defteam").agg(dropbacks_faced=("epa", "size"))
pass_d["pass_epa"] = wmean(pass_plays, "defteam")
pass_d["pass_success"] = wmean(pass_plays, "defteam", "success")
pass_d["adj_pass_epa"] = adj_def_pass
pass_d["explosive_pass_rate"] = wmean(pass_plays.assign(exp=pass_plays["yards_gained"] >= 20), "defteam", "exp")

# Coverage: EPA on actual throws (no sacks, no INTs), plus PFR per-target stats
throws = pass_plays[(pass_plays["sack"] == 0) & (pass_plays["interception"] == 0)]
pass_d["coverage_epa"] = wmean(throws, "defteam")
pass_d["yds_per_target"] = pfr["def_yards_allowed"] / pfr["def_targets"]
pass_d["cmp_pct_allowed"] = pfr["def_completions_allowed"] / pfr["def_targets"]

# Pass rush: all game situations (pressure doesn't depend on score)
dropbacks_all = all_plays[all_plays["pass"] == 1]
db_faced_all = dropbacks_all.groupby("defteam").size()
pass_d["pressure_rate"] = pfr["def_pressures"] / db_faced_all
pass_d["sack_rate"] = dropbacks_all.groupby("defteam")["sack"].mean()
pass_d["qb_hit_rate"] = dropbacks_all.groupby("defteam")["qb_hit"].mean()

charted = dropbacks_all.merge(ftn, left_on=["game_id", "play_id"],
                              right_on=["nflverse_game_id", "nflverse_play_id"])
charted["blitz"] = charted["n_blitzers"] > 0
pass_d["blitz_rate"] = charted.groupby("defteam")["blitz"].mean()
pass_d["avg_rushers"] = charted.groupby("defteam")["n_pass_rushers"].mean()
pass_d["no_blitz_epa"] = charted[~charted["blitz"]].groupby("defteam")["epa"].mean()
pass_d["interceptions"] = dropbacks_all.groupby("defteam")["interception"].sum()

pass_d = pass_d.sort_values("adj_pass_epa")
pass_d.insert(0, "rank", range(1, len(pass_d) + 1))
pass_d.insert(1, "raw_rank", pass_d["pass_epa"].rank(method="min").astype(int))
pass_d.insert(2, "coverage_rank", pass_d["coverage_epa"].rank(method="min").astype(int))
pass_d.insert(3, "pass_rush_rank", pass_d["pressure_rate"].rank(ascending=False, method="min").astype(int))
pass_d.index.name = "team"

pass_out = pass_d.round(3).reset_index()
pass_out["through_week"] = WEEK

print(f"{SEASON} pass defense through week {WEEK} (opponent-adjusted)")
display(pass_out)
save_table(pass_out, f"def_pass_{SEASON}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Rush defense
# MAGIC Ranked by **opponent-adjusted EPA per designed run** (lower = better). QB scrambles count as pass plays.

# COMMAND ----------

rush_d = rush_plays.groupby("defteam").agg(rushes_faced=("epa", "size"))
rush_d["rush_epa"] = wmean(rush_plays, "defteam")
rush_d["rush_success"] = wmean(rush_plays, "defteam", "success")
rush_d["yds_per_carry"] = wmean(rush_plays, "defteam", "yards_gained")
rush_d["adj_rush_epa"] = adj_def_rush
rush_d["stuff_rate"] = wmean(rush_plays.assign(stuff=rush_plays["yards_gained"] <= 0), "defteam", "stuff")
rush_d["explosive_run_rate"] = wmean(rush_plays.assign(exp=rush_plays["yards_gained"] >= 10), "defteam", "exp")
early_rush = rush_plays[rush_plays["down"].isin([1, 2])]
rush_d["early_down_rush_epa"] = wmean(early_rush, "defteam")

# Box counts (FTN): ties to the light-box thesis
rush_box = rush_plays.merge(ftn, left_on=["game_id", "play_id"],
                            right_on=["nflverse_game_id", "nflverse_play_id"])
rush_d["avg_box"] = rush_box.groupby("defteam")["n_defense_box"].mean()
rush_d["light_box_rate"] = (rush_box.assign(light=rush_box["n_defense_box"] <= 6)
                            .groupby("defteam")["light"].mean())
rush_d["light_box_rush_epa"] = wmean(rush_box[rush_box["n_defense_box"] <= 6], "defteam")

# Tackling (PFR): missed tackle rate on all plays
rush_d["missed_tackle_pct"] = pfr["def_missed_tackles"] / (pfr["def_missed_tackles"] + pfr["def_tackles_combined"])

rush_d = rush_d.sort_values("adj_rush_epa")
rush_d.insert(0, "rank", range(1, len(rush_d) + 1))
rush_d.insert(1, "raw_rank", rush_d["rush_epa"].rank(method="min").astype(int))
rush_d.index.name = "team"

rush_out = rush_d.round(3).reset_index()
rush_out["through_week"] = WEEK

print(f"{SEASON} rush defense through week {WEEK} (opponent-adjusted)")
display(rush_out)
save_table(rush_out, f"def_rush_{SEASON}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Offensive rankings
# MAGIC `rank` = quality (opponent-adjusted), `raw_rank` = results (plain EPA), `forecast_rank` = prediction.
# MAGIC Higher EPA = better. Includes adjusted pass and rush splits and run rate.

# COMMAND ----------

offense = plays.groupby("posteam").agg(
    plays=("epa", "size"),
    epa_per_play=("epa", "mean"),          # plain EPA, every play (Results view)
)
offense["weighted_epa"] = wmean(plays, "posteam")
offense["success_rate"] = wmean(plays, "posteam", "success")
offense["explosive_rate"] = wmean(plays, "posteam", "explosive")
offense["adj_epa_per_play"] = adj_off_epa
offense["forecast_score"] = off_forecast   # centered on league average; higher = better
offense["prior_epa"] = off_prior_epa
offense["prior_weight"] = off_prior_wt
offense["adj_pass_epa"] = adj_off_pass
offense["adj_rush_epa"] = adj_off_rush
offense["pass_rank"] = offense["adj_pass_epa"].rank(ascending=False, method="min").astype(int)
offense["rush_rank"] = offense["adj_rush_epa"].rank(ascending=False, method="min").astype(int)
offense["dropback_epa"] = wmean(pass_plays, "posteam")
offense["rush_epa"] = wmean(rush_plays, "posteam")
offense["dropbacks"] = pass_plays.groupby("posteam").size()
offense["rushes"] = rush_plays.groupby("posteam").size()
offense["run_rate"] = offense["rushes"] / (offense["rushes"] + offense["dropbacks"])   # team identity
offense["early_down_epa"] = wmean(early, "posteam")
offense["early_down_pass_rate"] = wmean(early, "posteam", "pass")
offense["series_conv_rate"] = off_series

db_off = all_plays[all_plays["pass"] == 1]
offense["sack_rate_allowed"] = db_off.groupby("posteam")["sack"].mean()
offense["qb_hit_rate_allowed"] = db_off.groupby("posteam")["qb_hit"].mean()
offense["giveaways"] = (all_plays.groupby("posteam")["interception"].sum()
                        + all_plays.groupby("posteam")["fumble_lost"].sum())
offense["penalties_pg"] = off_pen_pg
offense["penalty_yds_pg"] = off_pen_yds_pg

offense = offense.sort_values("adj_epa_per_play", ascending=False)
offense.insert(0, "rank", range(1, len(offense) + 1))
offense.insert(1, "raw_rank", offense["epa_per_play"].rank(ascending=False, method="min").astype(int))
offense.insert(2, "forecast_rank", offense["forecast_score"].rank(ascending=False, method="min").astype(int))
offense = offense.join(rank_range(plays, "posteam", offense["adj_epa_per_play"], ascending=False))
offense.index.name = "team"

off_out = offense.round(3).reset_index()
off_out["through_week"] = WEEK

print(f"{SEASON} offensive rankings through week {WEEK}")
display(off_out)
save_table(off_out, f"off_rankings_{SEASON}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Offense vs. defense efficiency
# MAGIC Top-right = good at both. Uses opponent-adjusted EPA; raw columns are saved too.

# COMMAND ----------

eff = pd.DataFrame({
    "off_epa": offense["epa_per_play"],
    "def_epa": defense["epa_per_play"],
    "adj_off_epa": offense["adj_epa_per_play"],
    "adj_def_epa": defense["adj_epa_per_play"],
    "off_success": offense["success_rate"],
    "def_success": defense["success_rate"],
})
eff["def_score"] = -eff["def_epa"]            # flipped so higher = better defense
eff["adj_def_score"] = -eff["adj_def_epa"]
eff["def_success_score"] = -eff["def_success"]
eff["net_adj_epa"] = eff["adj_off_epa"] - eff["adj_def_epa"]   # overall team strength
eff.index.name = "team"

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(eff["adj_off_epa"], eff["adj_def_score"])
for team, row in eff.iterrows():
    ax.annotate(team, (row["adj_off_epa"], row["adj_def_score"]), fontsize=8,
                xytext=(4, 4), textcoords="offset points")
ax.axvline(eff["adj_off_epa"].mean(), ls="--", c="gray")
ax.axhline(eff["adj_def_score"].mean(), ls="--", c="gray")
ax.set_xlabel("Opponent-adjusted offensive EPA/play (right = better)")
ax.set_ylabel("Opponent-adjusted defensive score (up = better)")
ax.set_title(f"{SEASON} team efficiency through week {WEEK} (garbage time down-weighted)")
plt.show()

eff_out = eff.sort_values("net_adj_epa", ascending=False).round(3).reset_index()
eff_out["through_week"] = WEEK
save_table(eff_out, f"team_efficiency_{SEASON}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. Game-by-game table (powers the interactive dashboard charts)
# MAGIC One row per team per game: offense and defense in the same row, so the dashboard
# MAGIC can filter by team and week and draw week-by-week trend lines.

# COMMAND ----------

gw = plays.assign(ew=plays["epa"] * plays["w"], sw=plays["success"] * plays["w"])
off_g = (gw.groupby(["posteam", "week", "defteam"])
         .agg(ew=("ew", "sum"), sw=("sw", "sum"), wt=("w", "sum"), off_plays=("epa", "size"))
         .reset_index().rename(columns={"posteam": "team", "defteam": "opponent"}))
off_g["off_epa"], off_g["off_success"] = off_g["ew"] / off_g["wt"], off_g["sw"] / off_g["wt"]
off_g = off_g.drop(columns=["ew", "sw", "wt"])
def_g = (gw.groupby(["defteam", "week", "posteam"])
         .agg(ew=("ew", "sum"), sw=("sw", "sum"), wt=("w", "sum"), def_plays=("epa", "size"))
         .reset_index().rename(columns={"defteam": "team", "posteam": "opponent"}))
def_g["def_epa"], def_g["def_success"] = def_g["ew"] / def_g["wt"], def_g["sw"] / def_g["wt"]
def_g = def_g.drop(columns=["ew", "sw", "wt"])

team_games = off_g.merge(def_g, on=["team", "week", "opponent"], how="outer")
adj_g = g[["posteam", "defteam", "week", "off_adj", "def_adj"]]
team_games = (team_games
              .merge(adj_g[["posteam", "defteam", "week", "off_adj"]]
                     .rename(columns={"posteam": "team", "defteam": "opponent"}),
                     on=["team", "opponent", "week"], how="left")
              .merge(adj_g[["defteam", "posteam", "week", "def_adj"]]
                     .rename(columns={"defteam": "team", "posteam": "opponent"}),
                     on=["team", "opponent", "week"], how="left"))
team_games["adj_off_epa"] = team_games["off_adj"] + league_epa
team_games["adj_def_epa"] = team_games["def_adj"] + league_epa
team_games = team_games.drop(columns=["off_adj", "def_adj"]).sort_values(["team", "week"])

# 3-game rolling form for trend lines
for c in ["off_epa", "def_epa", "adj_off_epa", "adj_def_epa"]:
    team_games[f"{c}_roll3"] = (team_games.groupby("team")[c]
                                .transform(lambda s: s.rolling(3, min_periods=1).mean()))
team_games["def_score_roll3"] = -team_games["def_epa_roll3"]

tg_out = team_games.round(3).reset_index(drop=True)
display(tg_out)
save_table(tg_out, f"team_games_{SEASON}")

In [0]:
## Personnel trend: base vs. nickel vs. dime (2016–2025) Core chart for the thesis. Participation data isn't released for 2026 yet. Regular season only. Note: data before 2023 comes from NFL Next Gen Stats and from 2023 on from FTN, so jumps right at 2023 likely reflect labeling differences, not real changes in how teams play.

import pandas as pd
import matplotlib.pyplot as plt

part = spark.sql("""
    select cast(substr(p.nflverse_game_id, 1, 4) as int) as season,
           p.defense_personnel
    from workspace.raw.participation p
    join workspace.raw.pbp b
      on p.nflverse_game_id = b.game_id and p.play_id = b.play_id
    where b.play_type in ('pass', 'run') and b.season_type = 'REG'
""").toPandas()

DB_POSITIONS = {"CB", "FS", "SS", "DB", "S"}

def count_dbs(personnel):
    """'3 CB, 2 DE, 2 DT, 1 FS, 2 ILB, 1 SS' -> 5 defensive backs"""
    if not isinstance(personnel, str):
        return None
    total = 0
    for chunk in personnel.split(","):
        parts = chunk.strip().split(" ")
        if len(parts) == 2 and parts[1] in DB_POSITIONS:
            total += int(parts[0])
    return total

part["n_db"] = part["defense_personnel"].apply(count_dbs)
part = part.dropna(subset=["n_db"])
part["package"] = pd.cut(part["n_db"], [-1, 4, 5, 11],
                         labels=["Base (4 or fewer DB)", "Nickel (5 DB)", "Dime+ (6+ DB)"])

usage = part.groupby("season")["package"].value_counts(normalize=True).unstack()
display(usage.round(3).reset_index())

usage.plot(marker="o", figsize=(10, 6), title="Share of defensive snaps by personnel package")
plt.ylabel("Share of snaps")
plt.xlabel("Season")
plt.show()

Research-Driven Questions: 

In [0]:
## Research question: the light-box tradeoff **Question:** Did defenses make the right call going lighter? What does a light box (6 or fewer defenders in the box) cost against the run, and what does it buy against the pass? **Method:** compare light vs. heavy boxes within the same season, down, and distance, so the situation doesn't bias the result, and so the 2023 NGS → FTN data-source change can't distort it. Regular season, early downs, competitive games. Saves `light_box_tradeoff`, `light_box_by_defense`, and `light_box_by_offense` to `workspace.analytics`.

# Build the play-level table: 2016–2025 plays joined to box counts
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def lb_prep(df):
    """Early downs, competitive game states, light/heavy box label, distance bucket."""
    df = df[df["down"].isin([1, 2]) & df["wp"].between(0.10, 0.90) & df["box_count"].notna()].copy()
    df["box"] = np.where(df["box_count"] <= 6, "light", "heavy")
    df["dist"] = pd.cut(df["ydstogo"], [0, 3, 7, 10, 99], labels=["1-3", "4-7", "8-10", "11+"])
    return df


lb_cols = "p.season, p.game_id, p.play_id, p.posteam, p.defteam, p.down, p.ydstogo, p.wp, p.pass, p.rush, p.epa, p.success"
lb_hist = spark.sql(f"""
    select {lb_cols}, pt.defenders_in_box as box_count
    from workspace.raw.pbp p
    join workspace.raw.participation pt
      on pt.nflverse_game_id = p.game_id and pt.play_id = p.play_id
    where p.season_type = 'REG' and (p.pass = 1 or p.rush = 1)
      and p.epa is not null and coalesce(p.two_point_attempt, 0) != 1
""").toPandas()
lb_hist = lb_prep(lb_hist)
print(f"{len(lb_hist):,} early-down plays with box counts, {lb_hist['season'].min()}–{lb_hist['season'].max()}")

In [0]:
### League answer: cost vs. benefit, by season `run_cost` = extra EPA offenses gain running against a light box. `pass_benefit` = EPA the defense saves on passes with a light box. `net_for_defense` below zero = the light box paid off.

def light_minus_heavy(df, by=("season",)):
    """EPA(light box) - EPA(heavy box), compared inside each down & distance bucket, then averaged."""
    t = (df.groupby(list(by) + ["down", "dist", "box"], observed=True)["epa"]
           .agg(["mean", "size"]).unstack("box").dropna())
    diff = t[("mean", "light")] - t[("mean", "heavy")]
    wt = np.minimum(t[("size", "light")], t[("size", "heavy")])   # weight buckets by usable sample
    lv = list(range(len(by)))
    return (diff * wt).groupby(level=lv).sum() / wt.groupby(level=lv).sum()


lb_runs, lb_passes = lb_hist[lb_hist["rush"] == 1], lb_hist[lb_hist["pass"] == 1]
tradeoff = pd.DataFrame({
    "run_cost": light_minus_heavy(lb_runs),
    "pass_benefit": -light_minus_heavy(lb_passes),
    "pass_rate_vs_light": lb_hist[lb_hist["box"] == "light"].groupby("season")["pass"].mean(),
    "light_box_share": (lb_hist["box"] == "light").groupby(lb_hist["season"]).mean(),
    "pass_epa": lb_passes.groupby("season")["epa"].mean(),
    "run_epa_vs_light": lb_runs[lb_runs["box"] == "light"].groupby("season")["epa"].mean(),
})
tradeoff["net_for_defense"] = ((1 - tradeoff["pass_rate_vs_light"]) * tradeoff["run_cost"]
                               - tradeoff["pass_rate_vs_light"] * tradeoff["pass_benefit"])
tradeoff.index.name = "season"
display(tradeoff.round(3).reset_index())
save_table(tradeoff.round(4).reset_index(), "light_box_tradeoff")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(tradeoff.index, tradeoff["run_cost"], marker="o", label="Cost: extra EPA per RUN vs light box")
ax.plot(tradeoff.index, tradeoff["pass_benefit"], marker="o", label="Benefit: EPA saved per PASS vs light box")
ax.bar(tradeoff.index, -tradeoff["net_for_defense"], alpha=0.3, label="Net for defense (up = light box paid off)")
ax.axhline(0, c="gray", lw=0.8)
ax.axvline(2022.5, ls=":", c="gray")
ax.text(2022.6, ax.get_ylim()[1] * 0.9, "data source: NGS → FTN", fontsize=8, color="gray")
ax.set_title("The light-box tradeoff, early downs (same down & distance)")
ax.set_ylabel("EPA per play")
ax.legend(fontsize=8)
plt.show()

In [0]:
### Which defenses win the tradeoff? (2022–2026, FTN box counts). FTN charting is one consistent source for 2022–2026 and updates during the season, so this includes the current year. Small samples: check `light_runs` and `light_passes`. `light_box_edge` below zero = the defense is better with a light box.

lb_ftn = spark.sql(f"""
    select {lb_cols}, f.n_defense_box as box_count
    from workspace.raw.pbp p
    join workspace.raw.ftn_charting f
      on f.nflverse_game_id = p.game_id and f.nflverse_play_id = p.play_id
    where p.season_type = 'REG' and (p.pass = 1 or p.rush = 1)
      and p.epa is not null and coalesce(p.two_point_attempt, 0) != 1
""").toPandas()
lb_ftn = lb_prep(lb_ftn)

lb_grp = ["season", "defteam"]
lb_r, lb_p = lb_ftn[lb_ftn["rush"] == 1], lb_ftn[lb_ftn["pass"] == 1]
lb_def = pd.DataFrame({
    "light_box_rate": (lb_ftn["box"] == "light").groupby([lb_ftn["season"], lb_ftn["defteam"]]).mean(),
    "run_epa_light": lb_r[lb_r["box"] == "light"].groupby(lb_grp)["epa"].mean(),
    "run_epa_heavy": lb_r[lb_r["box"] == "heavy"].groupby(lb_grp)["epa"].mean(),
    "pass_epa_light": lb_p[lb_p["box"] == "light"].groupby(lb_grp)["epa"].mean(),
    "pass_epa_heavy": lb_p[lb_p["box"] == "heavy"].groupby(lb_grp)["epa"].mean(),
    "light_runs": lb_r[lb_r["box"] == "light"].groupby(lb_grp).size(),
    "light_passes": lb_p[lb_p["box"] == "light"].groupby(lb_grp).size(),
})
lb_def["epa_allowed_light"] = lb_ftn[lb_ftn["box"] == "light"].groupby(lb_grp)["epa"].mean()
lb_def["epa_allowed_heavy"] = lb_ftn[lb_ftn["box"] == "heavy"].groupby(lb_grp)["epa"].mean()
lb_def["light_box_edge"] = lb_def["epa_allowed_light"] - lb_def["epa_allowed_heavy"]
lb_def = lb_def.reset_index().rename(columns={"defteam": "team"})
lb_def["light_edge_rank"] = lb_def.groupby("season")["light_box_edge"].rank(method="min").astype(int)

display(lb_def[lb_def["season"] == lb_def["season"].max()].sort_values("light_edge_rank").round(3))
save_table(lb_def.round(4), "light_box_by_defense")

In [0]:
### Which offenses punish light boxes on the ground? (2022–2026)

lb_off = pd.DataFrame({
    "run_rate_vs_light": lb_ftn[lb_ftn["box"] == "light"].groupby(["season", "posteam"])["rush"].mean(),
    "run_epa_vs_light": lb_r[lb_r["box"] == "light"].groupby(["season", "posteam"])["epa"].mean(),
    "run_epa_vs_heavy": lb_r[lb_r["box"] == "heavy"].groupby(["season", "posteam"])["epa"].mean(),
    "light_runs": lb_r[lb_r["box"] == "light"].groupby(["season", "posteam"]).size(),
}).reset_index().rename(columns={"posteam": "team"})
lb_off["light_box_punish"] = lb_off["run_epa_vs_light"] - lb_off["run_epa_vs_heavy"]
lb_off["punish_rank"] = lb_off.groupby("season")["run_epa_vs_light"].rank(ascending=False, method="min").astype(int)

display(lb_off[lb_off["season"] == lb_off["season"].max()].sort_values("punish_rank").round(3))
save_table(lb_off.round(4), "light_box_by_offense")